In [1]:

import gc
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    brier_score_loss
)
from sklearn.model_selection import GroupKFold
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [2]:
# Paths


DATA_PATH = r"E:\UofT\05_Research\00_File\output\data\ml_oilgas_pollution_panel_2018_2024.csv"
BASE_OUTPUT_DIR = r"E:\UofT\05_Research\00_File\output\data"

PRED_DIR = os.path.join(BASE_OUTPUT_DIR, "predictions")
CAND_DIR = os.path.join(BASE_OUTPUT_DIR, "candidates")
DIAG_DIR = os.path.join(BASE_OUTPUT_DIR, "diagnostics")

for d in [PRED_DIR, CAND_DIR, DIAG_DIR]:
    os.makedirs(d, exist_ok=True)

# Preferred split if 2022 is the event period of interest
TRAIN_END_YEAR = 2020
VAL_YEAR = 2021
TEST_START_YEAR = 2022

# Candidate thresholds
STRICT_MIN_THRESHOLD = None
DISCOVERY_THRESHOLD = None

# Spatial CV settings
N_SPATIAL_FOLDS = 5
SPATIAL_BLOCK_SIZE_DEG = 0.20  # roughly ~20km in latitude; simple and transparent

TARGET_COL = "flare_label"
ID_COLS = ["grid_id", "year", "month", "date", "lon", "lat"]

print("DATA_PATH:", DATA_PATH)
print("BASE_OUTPUT_DIR:", BASE_OUTPUT_DIR)


DATA_PATH: E:\UofT\05_Research\00_File\output\data\ml_oilgas_pollution_panel_2018_2024.csv
BASE_OUTPUT_DIR: E:\UofT\05_Research\00_File\output\data


In [3]:
TEMP_PARQUET_DIR = os.path.join(BASE_OUTPUT_DIR, "ml_oilgas_pollution_chunks")

os.makedirs(TEMP_PARQUET_DIR, exist_ok=True)

# optional: clear old chunk files
for f in os.listdir(TEMP_PARQUET_DIR):
    if f.endswith(".parquet"):
        os.remove(os.path.join(TEMP_PARQUET_DIR, f))

# ------------------------------------------------------------------
# Stream CSV -> parquet chunks
# ------------------------------------------------------------------
chunk_size = 25_000
chunk_num = 0

for chunk in pd.read_csv(DATA_PATH, chunksize=chunk_size):
    # parse date
    if "date" in chunk.columns:
        chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")

    # drop rows missing core fields
    required_cols = ["grid_id", "year", "month", "date", "lon", "lat", TARGET_COL]
    chunk = chunk.dropna(subset=required_cols).copy()

    # stable types
    chunk["grid_id"] = chunk["grid_id"].astype("string")
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce", downcast="integer")
    chunk["month"] = pd.to_numeric(chunk["month"], errors="coerce", downcast="integer")
    chunk["lon"] = pd.to_numeric(chunk["lon"], errors="coerce", downcast="float")
    chunk["lat"] = pd.to_numeric(chunk["lat"], errors="coerce", downcast="float")
    chunk[TARGET_COL] = pd.to_numeric(chunk[TARGET_COL], errors="coerce", downcast="integer")

    # downcast everything else numeric
    for col in chunk.select_dtypes(include=["int64", "int32"]).columns:
        chunk[col] = pd.to_numeric(chunk[col], downcast="integer")

    for col in chunk.select_dtypes(include=["float64"]).columns:
        chunk[col] = pd.to_numeric(chunk[col], downcast="float")

    chunk = chunk.replace([np.inf, -np.inf], np.nan)

    out_path = os.path.join(TEMP_PARQUET_DIR, f"part_{chunk_num:04d}.parquet")
    chunk.to_parquet(out_path, index=False)

    print(f"Saved {out_path} with {len(chunk):,} rows")
    chunk_num += 1

print("Done writing parquet chunks.")


parquet_files = [
    os.path.join(TEMP_PARQUET_DIR, f)
    for f in os.listdir(TEMP_PARQUET_DIR)
    if f.endswith(".parquet")
]

df = pd.concat([pd.read_parquet(f) for f in sorted(parquet_files)], ignore_index=True)

print("Rows:", len(df))
print("Years:", int(df["year"].min()), "to", int(df["year"].max()))
print("Positive rate:", round(df[TARGET_COL].mean(), 6))
print("Memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))

Saved E:\UofT\05_Research\00_File\output\data\ml_oilgas_pollution_chunks\part_0000.parquet with 25,000 rows
Saved E:\UofT\05_Research\00_File\output\data\ml_oilgas_pollution_chunks\part_0001.parquet with 25,000 rows
Saved E:\UofT\05_Research\00_File\output\data\ml_oilgas_pollution_chunks\part_0002.parquet with 25,000 rows
Saved E:\UofT\05_Research\00_File\output\data\ml_oilgas_pollution_chunks\part_0003.parquet with 25,000 rows
Saved E:\UofT\05_Research\00_File\output\data\ml_oilgas_pollution_chunks\part_0004.parquet with 25,000 rows
Saved E:\UofT\05_Research\00_File\output\data\ml_oilgas_pollution_chunks\part_0005.parquet with 25,000 rows
Saved E:\UofT\05_Research\00_File\output\data\ml_oilgas_pollution_chunks\part_0006.parquet with 25,000 rows
Saved E:\UofT\05_Research\00_File\output\data\ml_oilgas_pollution_chunks\part_0007.parquet with 25,000 rows
Saved E:\UofT\05_Research\00_File\output\data\ml_oilgas_pollution_chunks\part_0008.parquet with 25,000 rows
Saved E:\UofT\05_Research\00

In [4]:
# feature sets


# Full candidate feature pool expected from the revised prep script
full_feature_candidates = [
    # spatial
    "lon", "lat",

    # raw bands
    "B4_med", "B8_med", "B11_med",

    # raw variables
    "ndvi", "ndbi", "ntl", "ntl_clean", "ntl_log", "lst_night_c", "s2_count", "cf_cvg",

    # engineered contemporaneous features
    "builtup_index",
    "nir_red_ratio", "swir_nir_ratio", "swir_red_ratio",
    "heat_light_interaction", "built_light_interaction", "heat_built_interaction",
    "oil_signature_score",

    # anomalies
    "ntl_anom", "lst_anom", "ndvi_anom", "ndbi_anom", "b11_anom", "swir_nir_anom",

    # lagged levels
    "ntl_lag1", "lst_lag1", "b11_lag1", "ndvi_lag1", "ndbi_lag1", "swir_nir_lag1",

    # lagged rolling features
    "ntl_roll3", "ntl_roll6", "lst_roll3", "lst_roll6", "b11_roll3", "swir_nir_roll3",

    # changes
    "ntl_change", "lst_change", "b11_change", "ndvi_change", "ndbi_change", "swir_nir_change",

    # lagged persistence
    "ntl_persist6", "lst_persist6", "b11_persist6", "swir_nir_persist6",

    # seasonality
    "month_sin", "month_cos",

    # ---------------------------------
    # pollution: raw
    "CO", "NO2", "O3", "PM2_5", "SO2",

    # pollution: logs
    "CO_log", "NO2_log", "O3_log", "PM25_log", "SO2_log",

    # pollution: anomalies
    "CO_anom", "NO2_anom", "O3_anom", "PM25_anom", "SO2_anom",

    # pollution: lagged rolling
    "CO_roll3", "NO2_roll3", "O3_roll3", "PM25_roll3", "SO2_roll3",

    # pollution: changes
    "CO_change", "NO2_change", "O3_change", "PM25_change", "SO2_change",
]

# Missing flags produced by the revised R prep script
miss_flag_candidates = [c for c in df.columns if c.endswith("_miss")]

full_feature_cols = [c for c in full_feature_candidates if c in df.columns] + miss_flag_candidates
full_feature_cols = list(dict.fromkeys(full_feature_cols))

no_coord_feature_cols = [c for c in full_feature_cols if c not in {"lon", "lat"}]

reduced_feature_candidates = [
    "B4_med", "B8_med", "B11_med",
    "ndvi", "ndbi", "ntl_log", "lst_night_c",
    "builtup_index", "swir_nir_ratio", "swir_red_ratio",
    "ntl_anom", "lst_anom", "ndvi_anom", "ndbi_anom",
    "ntl_roll3", "ntl_roll6", "lst_roll3", "lst_roll6",
    "ntl_change", "lst_change",
    "ntl_persist6", "lst_persist6",
    "month_sin", "month_cos",

    # pollution
    "CO_log", "NO2_log", "O3_log", "PM25_log", "SO2_log",
    "CO_anom", "NO2_anom", "PM25_anom", "SO2_anom",
    "CO_roll3", "NO2_roll3", "PM25_roll3", "SO2_roll3",
    "CO_change", "NO2_change", "PM25_change", "SO2_change",
]
reduced_feature_cols = [c for c in reduced_feature_candidates if c in df.columns]
reduced_feature_cols += [c for c in miss_flag_candidates if c.replace("_miss", "") in reduced_feature_cols]
reduced_feature_cols = list(dict.fromkeys(reduced_feature_cols))

feature_sets = {
    "full": full_feature_cols,
    "no_coordinates": no_coord_feature_cols,
    "reduced": reduced_feature_cols,
}

for name, cols in feature_sets.items():
    print(name, "features:", len(cols))


full features: 114
no_coordinates features: 112
reduced features: 63


In [5]:
# Split definitions


train_mask = df["year"] <= TRAIN_END_YEAR
val_mask = df["year"] == VAL_YEAR
test_mask = df["year"] >= TEST_START_YEAR

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum())],
    "positive_rate": [
        df.loc[train_mask, TARGET_COL].mean(),
        df.loc[val_mask, TARGET_COL].mean(),
        df.loc[test_mask, TARGET_COL].mean()
    ]
})
split_summary


,split,rows,positive_rate
0,train,1816920,0.002417
1,validation,605640,0.002417
2,test,1816920,0.002417


In [6]:

def make_hgb_pipeline():
    return HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.05,
        max_depth=6,
        min_samples_leaf=40,
        l2_regularization=1.0,
        random_state=42
    )

def make_logit_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

def fit_pipeline_with_weights(model_obj, X, y):
    n_pos = int(y.sum())
    n_neg = int(len(y) - n_pos)
    pos_weight = (n_neg / max(n_pos, 1))
    sample_weight = np.where(y.to_numpy() == 1, pos_weight, 1.0)

    model = clone(model_obj)

    if hasattr(model, "named_steps") and "clf" in model.named_steps:
        model.fit(X, y, clf__sample_weight=sample_weight)
    else:
        model.fit(X, y, sample_weight=sample_weight)

    return model, pos_weight

df = df.replace([np.inf, -np.inf], np.nan)

def choose_best_threshold(y_true, pred_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, pred_prob)
    f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = int(np.argmax(f1_scores))
    return {
        "best_threshold": float(thresholds[best_idx]),
        "best_f1": float(f1_scores[best_idx])
    }

def evaluate_predictions(y_true, pred_prob, threshold):
    pred_label = (pred_prob >= threshold).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, pred_prob)),
        "pr_auc": float(average_precision_score(y_true, pred_prob)),
        "brier": float(brier_score_loss(y_true, pred_prob)),
        "precision": float(precision_score(y_true, pred_label, zero_division=0)),
        "recall": float(recall_score(y_true, pred_label, zero_division=0)),
        "f1": float(f1_score(y_true, pred_label, zero_division=0)),
        "predicted_positives": int(pred_label.sum())
    }

def calibrate_isotonic(y_val, pred_val, pred_to_calibrate):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(pred_val, y_val)
    calibrated = iso.transform(pred_to_calibrate)
    return calibrated, iso

def make_spatial_groups(data, block_size_deg=0.20):
    lon_block = np.floor(data["lon"] / block_size_deg).astype(int)
    lat_block = np.floor(data["lat"] / block_size_deg).astype(int)
    return lon_block.astype(str) + "_" + lat_block.astype(str)

def run_spatial_block_cv(data, feature_cols, pipeline, n_splits=5, pre_event_end_year=2021):
    cv_data = data.loc[data["year"] <= pre_event_end_year].copy()
    groups = make_spatial_groups(cv_data, block_size_deg=SPATIAL_BLOCK_SIZE_DEG)
    unique_groups = pd.Series(groups).nunique()
    n_splits = min(n_splits, unique_groups)
    if n_splits < 2:
        return pd.DataFrame()

    gkf = GroupKFold(n_splits=n_splits)
    X = cv_data[feature_cols]
    y = cv_data[TARGET_COL]

    rows = []
    for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
        X_tr = X.iloc[tr_idx]
        y_tr = y.iloc[tr_idx]
        X_te = X.iloc[te_idx]
        y_te = y.iloc[te_idx]

        model, _ = fit_pipeline_with_weights(pipeline, X_tr, y_tr)
        pred = model.predict_proba(X_te)[:, 1]

        rows.append({
            "fold": fold,
            "rows_test": len(te_idx),
            "positive_rate_test": float(y_te.mean()),
            "roc_auc": float(roc_auc_score(y_te, pred)),
            "pr_auc": float(average_precision_score(y_te, pred)),
            "brier": float(brier_score_loss(y_te, pred)),
        })

    return pd.DataFrame(rows)

def score_in_chunks(model, data, feature_cols, chunk_size=300000):
    pred_parts = []
    for start in range(0, len(data), chunk_size):
        end = min(start + chunk_size, len(data))
        X_chunk = data.iloc[start:end][feature_cols]
        pred_chunk = model.predict_proba(X_chunk)[:, 1]
        pred_parts.append(pred_chunk)
        print(f"Scored rows {start:,} to {end:,}")
    return np.concatenate(pred_parts)


In [7]:

# ------------------------------------------------------------
# Train and compare HGB models across feature sets
# ------------------------------------------------------------
results = []
fitted_models = {}
validation_predictions = {}
test_predictions = {}
calibrators = {}

base_pipeline = make_hgb_pipeline()

for spec_name, feature_cols in feature_sets.items():
    print(f"\n===== Training spec: {spec_name} =====")

    X_train = df.loc[train_mask, feature_cols]
    y_train = df.loc[train_mask, TARGET_COL]
    X_val = df.loc[val_mask, feature_cols]
    y_val = df.loc[val_mask, TARGET_COL]
    X_test = df.loc[test_mask, feature_cols]
    y_test = df.loc[test_mask, TARGET_COL]

    model, pos_weight = fit_pipeline_with_weights(base_pipeline, X_train, y_train)

    pred_val_raw = model.predict_proba(X_val)[:, 1]
    pred_val_cal, iso = calibrate_isotonic(y_val, pred_val_raw, pred_val_raw)

    threshold_info = choose_best_threshold(y_val, pred_val_cal)

    pred_test_raw = model.predict_proba(X_test)[:, 1]
    pred_test_cal = iso.transform(pred_test_raw)

    val_metrics_raw = evaluate_predictions(y_val, pred_val_raw, threshold_info["best_threshold"])
    val_metrics_cal = evaluate_predictions(y_val, pred_val_cal, threshold_info["best_threshold"])
    test_metrics_raw = evaluate_predictions(y_test, pred_test_raw, threshold_info["best_threshold"])
    test_metrics_cal = evaluate_predictions(y_test, pred_test_cal, threshold_info["best_threshold"])

    results.append({
        "model": "HGB",
        "spec": spec_name,
        "n_features": len(feature_cols),
        "pos_weight": pos_weight,
        "threshold_validation_f1": threshold_info["best_threshold"],
        "validation_f1_at_threshold": threshold_info["best_f1"],
        "val_pr_auc_raw": val_metrics_raw["pr_auc"],
        "val_pr_auc_cal": val_metrics_cal["pr_auc"],
        "val_roc_auc_raw": val_metrics_raw["roc_auc"],
        "val_brier_raw": val_metrics_raw["brier"],
        "val_brier_cal": val_metrics_cal["brier"],
        "test_pr_auc_raw": test_metrics_raw["pr_auc"],
        "test_pr_auc_cal": test_metrics_cal["pr_auc"],
        "test_roc_auc_raw": test_metrics_raw["roc_auc"],
        "test_brier_raw": test_metrics_raw["brier"],
        "test_brier_cal": test_metrics_cal["brier"],
        "test_precision_at_val_threshold_raw": test_metrics_raw["precision"],
        "test_recall_at_val_threshold_raw": test_metrics_raw["recall"],
        "test_f1_at_val_threshold_raw": test_metrics_raw["f1"],
        "test_precision_at_val_threshold_cal": test_metrics_cal["precision"],
        "test_recall_at_val_threshold_cal": test_metrics_cal["recall"],
        "test_f1_at_val_threshold_cal": test_metrics_cal["f1"]
    })

    fitted_models[spec_name] = {
        "model": model,
        "feature_cols": feature_cols,
        "threshold": threshold_info["best_threshold"],
        "iso": iso
    }
    validation_predictions[spec_name] = pred_val_cal
    test_predictions[spec_name] = pred_test_cal
    calibrators[spec_name] = iso

results_df = pd.DataFrame(results).sort_values(
    ["val_pr_auc_cal", "val_roc_auc_raw"],
    ascending=False
)

results_df



===== Training spec: full =====

===== Training spec: no_coordinates =====

===== Training spec: reduced =====


,model,spec,n_features,pos_weight,threshold_validation_f1,validation_f1_at_threshold,val_pr_auc_raw,val_pr_auc_cal,val_roc_auc_raw,val_brier_raw,val_brier_cal,test_pr_auc_raw,test_pr_auc_cal,test_roc_auc_raw,test_brier_raw,test_brier_cal,test_precision_at_val_threshold_raw,test_recall_at_val_threshold_raw,test_f1_at_val_threshold_raw,test_precision_at_val_threshold_cal,test_recall_at_val_threshold_cal,test_f1_at_val_threshold_cal
0,HGB,full,114,412.688525,0.428571,0.738703,0.782616,0.775003,0.999176,0.007084,0.000934,0.750980,0.734171,0.998958,0.005876,0.001054,0.209142,0.983379,0.344927,0.701763,0.688980,0.695312
1,HGB,no_coordinates,112,412.688525,0.265625,0.432863,0.342122,0.338057,0.985745,0.026198,0.001838,0.294632,0.282062,0.973535,0.021574,0.001960,0.031350,0.877732,0.060538,0.483436,0.285747,0.359187
2,HGB,reduced,63,412.688525,0.324444,0.424289,0.330454,0.326564,0.983863,0.027171,0.001853,0.293458,0.279344,0.973826,0.023387,0.001962,0.036554,0.862250,0.070135,0.461622,0.291667,0.357472


In [8]:

# ------------------------------------------------------------
# Optional benchmark model: logistic regression on reduced set
# ------------------------------------------------------------
benchmark_rows = []

benchmark_spec = "reduced"
benchmark_features = feature_sets[benchmark_spec]

X_train = df.loc[train_mask, benchmark_features]
y_train = df.loc[train_mask, TARGET_COL]
X_val = df.loc[val_mask, benchmark_features]
y_val = df.loc[val_mask, TARGET_COL]
X_test = df.loc[test_mask, benchmark_features]
y_test = df.loc[test_mask, TARGET_COL]

logit_model, _ = fit_pipeline_with_weights(make_logit_pipeline(), X_train, y_train)

pred_val_raw = logit_model.predict_proba(X_val)[:, 1]
pred_val_cal, logit_iso = calibrate_isotonic(y_val, pred_val_raw, pred_val_raw)
threshold_info = choose_best_threshold(y_val, pred_val_cal)

pred_test_raw = logit_model.predict_proba(X_test)[:, 1]
pred_test_cal = logit_iso.transform(pred_test_raw)


benchmark_rows.append({
    "model": "Logistic",
    "spec": benchmark_spec,
    "n_features": len(benchmark_features),
    "threshold_validation_f1": threshold_info["best_threshold"],
    "validation_f1_at_threshold": threshold_info["best_f1"],
    "val_pr_auc_cal": average_precision_score(y_val, pred_val_cal),
    "test_pr_auc_cal": average_precision_score(y_test, pred_test_cal),
    "test_roc_auc_raw": roc_auc_score(y_test, pred_test_raw),
    "test_brier_cal": brier_score_loss(y_test, pred_test_cal)
})

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df


,model,spec,n_features,threshold_validation_f1,validation_f1_at_threshold,val_pr_auc_cal,test_pr_auc_cal,test_roc_auc_raw,test_brier_cal
0,Logistic,reduced,63,0.222222,0.38751,0.271105,0.277734,0.972633,0.001931


In [9]:

# ------------------------------------------------------------
# Spatial block CV diagnostic on pre-event years only
# ------------------------------------------------------------
spatial_cv_rows = []

for spec_name, feature_cols in feature_sets.items():
    cv_df = run_spatial_block_cv(
        data=df,
        feature_cols=feature_cols,
        pipeline=make_hgb_pipeline(),
        n_splits=N_SPATIAL_FOLDS,
        pre_event_end_year=TRAIN_END_YEAR
    )

    if len(cv_df) > 0:
        spatial_cv_rows.append({
            "spec": spec_name,
            "mean_pr_auc": cv_df["pr_auc"].mean(),
            "mean_roc_auc": cv_df["roc_auc"].mean(),
            "mean_brier": cv_df["brier"].mean(),
            "folds": len(cv_df)
        })

spatial_cv_summary = pd.DataFrame(spatial_cv_rows).sort_values("mean_pr_auc", ascending=False)
spatial_cv_summary


,spec,mean_pr_auc,mean_roc_auc,mean_brier,folds
0,full,0.270597,0.948916,0.006770,5
1,no_coordinates,0.270548,0.946343,0.022559,5
2,reduced,0.266669,0.947407,0.023269,5


In [10]:
selection_df = (
    results_df[[
        "spec",
        "val_pr_auc_cal",
        "val_roc_auc_raw"
    ]]
    .merge(
        spatial_cv_summary.rename(columns={
            "mean_pr_auc": "spatial_pr_auc_mean",
            "mean_roc_auc": "spatial_roc_auc_mean",
            "mean_brier": "spatial_brier_mean"
        }),
        on="spec",
        how="left"
    )
    .copy()
)

# Safety: fill any missing spatial metrics with 0 so selection still runs
for col in ["spatial_pr_auc_mean", "spatial_roc_auc_mean", "spatial_brier_mean"]:
    if col in selection_df.columns:
        selection_df[col] = selection_df[col].fillna(0.0)

# ------------------------------------------------------------
# Normalize metrics so validation PR-AUC and spatial PR-AUC
# are compared on the same scale
# ------------------------------------------------------------
val_max = selection_df["val_pr_auc_cal"].max()
spatial_max = selection_df["spatial_pr_auc_mean"].max()

selection_df["val_pr_auc_norm"] = (
    selection_df["val_pr_auc_cal"] / val_max if val_max > 0 else 0.0
)

selection_df["spatial_pr_auc_norm"] = (
    selection_df["spatial_pr_auc_mean"] / spatial_max if spatial_max > 0 else 0.0
)

# ------------------------------------------------------------
# Selection score: put more weight on spatial generalization
# ------------------------------------------------------------
selection_df["selection_score"] = (
    0.4 * selection_df["val_pr_auc_norm"] +
    0.6 * selection_df["spatial_pr_auc_norm"]
)

# ------------------------------------------------------------
# Softer spatial screen (optional but more defensible)
# Keep models within 80% of the best spatial performer
# ------------------------------------------------------------
best_spatial = selection_df["spatial_pr_auc_mean"].max()
selection_df["passes_spatial"] = (
    selection_df["spatial_pr_auc_mean"] >= 0.80 * best_spatial
)

eligible_df = selection_df[selection_df["passes_spatial"]].copy()
if eligible_df.empty:
    eligible_df = selection_df.copy()

eligible_df = eligible_df.sort_values(
    [
        "selection_score",
        "spatial_pr_auc_norm",
        "val_pr_auc_norm",
        "spatial_pr_auc_mean",
        "val_pr_auc_cal"
    ],
    ascending=False
)

preferred_spec = eligible_df.iloc[0]["spec"]
preferred = fitted_models[preferred_spec]

print("Preferred spec:", preferred_spec)
print("Features:", len(preferred["feature_cols"]))
print("Validation threshold:", preferred["threshold"])

selection_df.sort_values(
    ["selection_score", "spatial_pr_auc_norm", "val_pr_auc_norm"],
    ascending=False
)

Preferred spec: full
Features: 114
Validation threshold: 0.42857142857142855


,spec,val_pr_auc_cal,val_roc_auc_raw,spatial_pr_auc_mean,spatial_roc_auc_mean,spatial_brier_mean,folds,val_pr_auc_norm,spatial_pr_auc_norm,selection_score,passes_spatial
0,full,0.775003,0.999176,0.270597,0.948916,0.006770,5,1.000000,1.000000,1.000000,True
1,no_coordinates,0.338057,0.985745,0.270548,0.946343,0.022559,5,0.436201,0.999818,0.774371,True
2,reduced,0.326564,0.983863,0.266669,0.947407,0.023269,5,0.421371,0.985485,0.759839,True


In [11]:

# ------------------------------------------------------------
# Final diagnostics for preferred model
# ------------------------------------------------------------
preferred_model = preferred["model"]
preferred_features = preferred["feature_cols"]
preferred_threshold = preferred["threshold"]
preferred_iso = preferred["iso"]

X_val = df.loc[val_mask, preferred_features]
y_val = df.loc[val_mask, TARGET_COL]
X_test = df.loc[test_mask, preferred_features]
y_test = df.loc[test_mask, TARGET_COL]

pred_val_raw = preferred_model.predict_proba(X_val)[:, 1]
pred_val_cal = preferred_iso.transform(pred_val_raw)

pred_test_raw = preferred_model.predict_proba(X_test)[:, 1]
pred_test_cal = preferred_iso.transform(pred_test_raw)

print("Validation metrics (calibrated):")
print(evaluate_predictions(y_val, pred_val_cal, preferred_threshold))

print("\nTest metrics (calibrated):")
print(evaluate_predictions(y_test, pred_test_cal, preferred_threshold))

pred_label_test = (pred_test_cal >= preferred_threshold).astype(int)
print("\nClassification report (test, calibrated probabilities, validation threshold):")
print(classification_report(y_test, pred_label_test, digits=3))

print("Confusion matrix:")
print(confusion_matrix(y_test, pred_label_test))


Validation metrics (calibrated):
{'roc_auc': 0.9992101156506271, 'pr_auc': 0.7750030072201383, 'brier': 0.0009337914514028456, 'precision': 0.7094339622641509, 'recall': 0.7704918032786885, 'f1': 0.7387033398821218, 'predicted_positives': 1590}

Test metrics (calibrated):
{'roc_auc': 0.9984858347096277, 'pr_auc': 0.734170666234966, 'brier': 0.0010542620092200581, 'precision': 0.7017625231910947, 'recall': 0.6889799635701275, 'f1': 0.6953125, 'predicted_positives': 4312}

Classification report (test, calibrated probabilities, validation threshold):
              precision    recall  f1-score   support

           0      0.999     0.999     0.999   1812528
           1      0.702     0.689     0.695      4392

    accuracy                          0.999   1816920
   macro avg      0.851     0.844     0.847   1816920
weighted avg      0.999     0.999     0.999   1816920

Confusion matrix:
[[1811242    1286]
 [   1366    3026]]


In [12]:

# ------------------------------------------------------------
# Permutation importance for preferred model
# ------------------------------------------------------------
sample_n = min(100000, int(test_mask.sum()))
rng = np.random.RandomState(42)
test_index = np.where(test_mask)[0]
sample_index = rng.choice(test_index, size=sample_n, replace=False)

X_test_sample = df.iloc[sample_index][preferred_features]
y_test_sample = df.iloc[sample_index][TARGET_COL]

perm = permutation_importance(
    preferred_model,
    X_test_sample,
    y_test_sample,
    n_repeats=5,
    random_state=42,
    scoring="average_precision"
)

perm_importance = (
    pd.DataFrame({
        "feature": preferred_features,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

perm_importance.head(25)


,feature,importance_mean,importance_std
0,lon,0.607792,0.023711
1,lat,0.524334,0.016106
2,ntl,0.219484,0.009528
3,ntl_roll6,0.215067,0.015428
4,ntl_anom,0.073100,0.005880
5,swir_red_ratio,0.068577,0.004662
6,ntl_roll3,0.055758,0.009824
7,NO2_roll3,0.040645,0.009506
8,ntl_persist6,0.017566,0.001721
9,ndbi_anom,0.013207,0.002468


In [13]:
# ------------------------------------------------------------
# Score full panel with preferred model and calibrator
# ------------------------------------------------------------
df_scored = df.copy()

pred_full_raw = score_in_chunks(preferred_model, df_scored, preferred_features, chunk_size=300000)
df_scored["pred_prob_raw"] = pred_full_raw
df_scored["pred_prob"] = preferred_iso.transform(pred_full_raw)

# Define post-event period
df_scored["post_event"] = df_scored["date"] >= "2022-02-01"

# Candidate thresholds based on the post-event score distribution
post_scores = df_scored.loc[df_scored["post_event"], "pred_prob"]

strict_threshold = float(post_scores.quantile(0.995))      # top 0.5%
discovery_threshold = float(post_scores.quantile(0.975))   # top 2.5%

# Flags
df_scored["active_flag_validation_f1"] = (df_scored["pred_prob"] >= preferred_threshold).astype("int8")
df_scored["active_flag_strict"] = (df_scored["pred_prob"] >= strict_threshold).astype("int8")
df_scored["active_flag_discovery"] = (df_scored["pred_prob"] >= discovery_threshold).astype("int8")

print("Preferred threshold :", preferred_threshold)
print("Strict threshold    :", strict_threshold)
print("Discovery threshold :", discovery_threshold)

df_scored[["grid_id", "year", "month", "pred_prob"]].head()

Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,300,000
Scored rows 3,300,000 to 3,600,000
Scored rows 3,600,000 to 3,900,000
Scored rows 3,900,000 to 4,200,000
Scored rows 4,200,000 to 4,239,480
Preferred threshold : 0.42857142857142855
Strict threshold    : 0.046413502109704644
Discovery threshold : 0.00017828489926903192


,grid_id,year,month,pred_prob
0,47252,2018,1,0.0
1,47065,2018,1,0.0
2,47439,2018,1,0.0
3,47253,2018,1,0.0
4,47066,2018,1,0.0


In [14]:
# ------------------------------------------------------------
# Aggregate to grid level
# ------------------------------------------------------------
agg_dict = {
    "lon": ("lon", "mean"),
    "lat": ("lat", "mean"),
    "flare_label_ever": (TARGET_COL, "max"),
    "mean_risk": ("pred_prob", "mean"),
    "max_risk": ("pred_prob", "max"),
    "active_months_validation_f1": ("active_flag_validation_f1", "sum"),
    "strict_active_months": ("active_flag_strict", "sum"),
    "discovery_active_months": ("active_flag_discovery", "sum"),

    # post-event metrics
    "post_max_risk": ("pred_prob", lambda x: x[df_scored.loc[x.index, "post_event"]].max() if df_scored.loc[x.index, "post_event"].any() else np.nan),
    "post_mean_risk": ("pred_prob", lambda x: x[df_scored.loc[x.index, "post_event"]].mean() if df_scored.loc[x.index, "post_event"].any() else np.nan),
    "post_strict_active_months": ("active_flag_strict", lambda x: x[df_scored.loc[x.index, "post_event"]].sum()),
    "post_discovery_active_months": ("active_flag_discovery", lambda x: x[df_scored.loc[x.index, "post_event"]].sum())
}

if "oilgas_candidate" in df_scored.columns:
    agg_dict["oilgas_candidate_ever"] = ("oilgas_candidate", "max")

grid_scores = (
    df_scored
    .groupby("grid_id", as_index=False)
    .agg(**agg_dict)
)

grid_scores.head()

,grid_id,lon,lat,flare_label_ever,mean_risk,max_risk,active_months_validation_f1,strict_active_months,discovery_active_months,post_max_risk,post_mean_risk,post_strict_active_months,post_discovery_active_months,oilgas_candidate_ever
0,1,45.212429,34.507957,0,0.000000,0.000000,0,0,0,0.0,0.0,0,0,0
1,10,45.245167,34.525932,0,0.000000,0.000000,0,0,0,0.0,0.0,0,0,0
2,100,45.267216,34.607048,0,0.000000,0.000000,0,0,0,0.0,0.0,0,0,1
3,1000,44.239944,34.830414,0,0.000000,0.000000,0,0,0,0.0,0.0,0,0,0
4,10000,44.444736,35.273365,0,0.000131,0.011015,0,0,1,0.0,0.0,0,0,1


In [15]:

# ------------------------------------------------------------
# Candidate sets
# ------------------------------------------------------------
strict_candidates = (
    grid_scores[
        (grid_scores["flare_label_ever"] == 0) &
        (grid_scores["post_strict_active_months"] >= 1)
    ]
    .sort_values(["post_max_risk", "post_strict_active_months"], ascending=[False, False])
    .copy()
)

discovery_candidates = (
    grid_scores[
        (grid_scores["flare_label_ever"] == 0) &
        (
            (grid_scores["post_discovery_active_months"] >= 2) |
            (grid_scores["post_mean_risk"] >= discovery_threshold)
        )
    ]
    .sort_values(["post_max_risk", "post_discovery_active_months"], ascending=[False, False])
    .copy()
)

if "oilgas_candidate_ever" in grid_scores.columns:
    strict_candidates_filtered = strict_candidates.loc[
        strict_candidates["oilgas_candidate_ever"] == 1
    ].copy()
else:
    strict_candidates_filtered = pd.DataFrame()

print("Strict candidates:", len(strict_candidates))
print("Discovery candidates:", len(discovery_candidates))
print("Strict candidates after heuristic filter:", len(strict_candidates_filtered))
strict_candidates.head(20)


Strict candidates: 792
Discovery candidates: 5413
Strict candidates after heuristic filter: 491


,grid_id,lon,lat,flare_label_ever,mean_risk,max_risk,active_months_validation_f1,strict_active_months,discovery_active_months,post_max_risk,post_mean_risk,post_strict_active_months,post_discovery_active_months,oilgas_candidate_ever
7769,16991,44.354771,35.525383,0,0.626228,1.000000,62,83,84,1.000000,0.700848,34,35,1
8038,17232,44.354698,35.534401,0,0.633581,0.945946,62,84,84,0.945946,0.766687,35,35,1
36272,42643,43.378056,36.904438,0,0.482082,0.945946,34,57,60,0.945946,0.641746,23,23,0
79545,9005,44.423000,35.237194,0,0.717294,0.945946,51,55,60,0.945946,0.513227,18,23,1
6690,16019,44.167652,35.488171,0,0.364087,0.942197,34,82,84,0.942197,0.482916,34,35,0
7502,16750,44.343815,35.516308,0,0.597876,0.942197,69,83,84,0.942197,0.608079,34,35,1
6419,15776,44.145702,35.479000,0,0.345026,0.942197,32,75,84,0.942197,0.360418,33,35,1
79546,9006,44.433990,35.237247,0,0.732127,0.945946,73,81,84,0.942197,0.609399,32,35,1
36073,42464,43.400684,36.895733,0,0.340558,0.942197,22,57,60,0.942197,0.577352,23,23,0
6421,15778,44.167747,35.479160,0,0.294434,0.942197,18,48,60,0.942197,0.488948,22,23,1


In [16]:

# ------------------------------------------------------------
# Build economics-ready panel outputs
# ------------------------------------------------------------
grid_month_predictions = df_scored[
    ["grid_id", "year", "month", "date", "lon", "lat", TARGET_COL, "pred_prob_raw", "pred_prob",
     "active_flag_validation_f1", "active_flag_strict", "active_flag_discovery"]
].copy()

# Tail indicators for econometric work
pre_event_mask = grid_month_predictions["date"] < "2022-02-01"

for q in [0.90, 0.95, 0.99]:
    cut = grid_month_predictions.loc[pre_event_mask, "pred_prob"].quantile(q)
    grid_month_predictions[f"high{int(q*100)}"] = (
        grid_month_predictions["pred_prob"] >= cut
    ).astype("int8")

# Simple monthly aggregates
monthly_risk = (
    grid_month_predictions
    .groupby(["year", "month", "date"], as_index=False)
    .agg(
        mean_risk=("pred_prob", "mean"),
        p95_risk=("pred_prob", lambda x: np.quantile(x, 0.95)),
        p99_risk=("pred_prob", lambda x: np.quantile(x, 0.99)),
        active_share=("active_flag_validation_f1", "mean"),
        strict_active_share=("active_flag_strict", "mean"),
        n_grids=("grid_id", "nunique")
    )
)

grid_month_predictions.head(), monthly_risk.head()


(  grid_id  year  month       date        lon        lat  flare_label  pred_prob_raw  pred_prob  active_flag_validation_f1  active_flag_strict  active_flag_discovery  high90  high95  high99
 0   47252  2018      1 2018-01-01  42.349930  37.102230            0       0.002634        0.0                          0                   0                      0       1       1       0
 1   47065  2018      1 2018-01-01  42.350243  37.093227            0       0.000612        0.0                          0                   0                      0       1       1       0
 2   47439  2018      1 2018-01-01  42.360859  37.111488            0       0.000365        0.0                          0                   0                      0       1       1       0
 3   47253  2018      1 2018-01-01  42.361172  37.102482            0       0.003716        0.0                          0                   0                      0       1       1       0
 4   47066  2018      1 2018-01-01  42.361481  37.

In [17]:

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
results_df.to_csv(os.path.join(DIAG_DIR, "model_comparison_results.csv"), index=False)
benchmark_df.to_csv(os.path.join(DIAG_DIR, "benchmark_logistic_results.csv"), index=False)
spatial_cv_summary.to_csv(os.path.join(DIAG_DIR, "spatial_cv_summary.csv"), index=False)
perm_importance.to_csv(os.path.join(DIAG_DIR, "preferred_model_permutation_importance.csv"), index=False)

df_scored.to_csv(os.path.join(PRED_DIR, "grid_month_predictions_scored_full.csv"), index=False)
grid_month_predictions.to_csv(os.path.join(PRED_DIR, "grid_month_predictions.csv"), index=False)
monthly_risk.to_csv(os.path.join(PRED_DIR, "monthly_risk_series.csv"), index=False)
grid_scores.to_csv(os.path.join(PRED_DIR, "grid_level_scores.csv"), index=False)

strict_candidates.to_csv(os.path.join(CAND_DIR, "strict_candidates.csv"), index=False)
discovery_candidates.to_csv(os.path.join(CAND_DIR, "discovery_candidates.csv"), index=False)
strict_candidates_filtered.to_csv(os.path.join(CAND_DIR, "strict_candidates_filtered.csv"), index=False)

print("Saved diagnostics to:", DIAG_DIR)
print("Saved predictions to:", PRED_DIR)
print("Saved candidates to:", CAND_DIR)


Saved diagnostics to: E:\UofT\05_Research\00_File\output\data\diagnostics
Saved predictions to: E:\UofT\05_Research\00_File\output\data\predictions
Saved candidates to: E:\UofT\05_Research\00_File\output\data\candidates


In [18]:

# ------------------------------------------------------------
# Compact paper summary tables to inspect quickly
# ------------------------------------------------------------
paper_model_table = results_df[[
    "model", "spec", "n_features",
    "val_pr_auc_cal", "test_pr_auc_cal",
    "test_roc_auc_raw", "test_brier_cal",
    "threshold_validation_f1",
    "test_precision_at_val_threshold_cal",
    "test_recall_at_val_threshold_cal",
    "test_f1_at_val_threshold_cal"
]].copy()

paper_model_table


,model,spec,n_features,val_pr_auc_cal,test_pr_auc_cal,test_roc_auc_raw,test_brier_cal,threshold_validation_f1,test_precision_at_val_threshold_cal,test_recall_at_val_threshold_cal,test_f1_at_val_threshold_cal
0,HGB,full,114,0.775003,0.734171,0.998958,0.001054,0.428571,0.701763,0.688980,0.695312
1,HGB,no_coordinates,112,0.338057,0.282062,0.973535,0.001960,0.265625,0.483436,0.285747,0.359187
2,HGB,reduced,63,0.326564,0.279344,0.973826,0.001962,0.324444,0.461622,0.291667,0.357472
